In [2]:
import pandas as pd
import os

In [8]:
df = pd.read_csv('../../data/raw_data/cardekho_listings_delhi.csv')

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7536 entries, 0 to 7535
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Title        7536 non-null   str  
 1   Link         7536 non-null   str  
 2   Price        7530 non-null   str  
 3   Information  7536 non-null   str  
 4   Image        7535 non-null   str  
 5   Year         7536 non-null   int64
 6   Brand        7536 non-null   str  
 7   Location     7536 non-null   str  
dtypes: int64(1), str(7)
memory usage: 471.1 KB


In [19]:
df.columns

Index(['Title', 'Link', 'Price', 'Information', 'Image', 'Year', 'Brand',
       'Location'],
      dtype='str')

In [14]:
df.head()

,Title,Link,Price,Information,Image,Year,Brand,Location
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,"₹ 231,000","60,000 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon"
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","58,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi"
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","41,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi"
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,"₹ 1,092,000","53,366 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad"
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,"₹ 830,000","1,46,000 kms • Diesel • Manual",https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida


In [37]:
units = set()

def handle_information(data):
    data = data.replace(' • ', '')
    parts = data.split()
    
    distance = parts[0].replace(',', '')
    unit = parts[1]
    units.add(unit)

    if len(parts) == 2:
        fuel_type,vehicle_type= None,None
    elif len(parts) == 3:
        fuel_type, vehicle_type = parts[2], None
    elif len(parts) == 4:
        fuel_type, vehicle_type = parts[2], parts[3]
    else:
        raise ValueError(f"Unexpected format: {parts}")
    
    return (int(distance), fuel_type, vehicle_type)

df[['Kms Covered', 'Fuel Type', 'Type']] = df['Information'].apply(handle_information).apply(pd.Series)

In [40]:
df.head()

,Title,Link,Price,Information,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,"₹ 231,000","60,000 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon",60000.0,Petrol,Automatic
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","58,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi",58000.0,Petrol,Manual
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","41,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi",41000.0,Petrol,Manual
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,"₹ 1,092,000","53,366 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad",53366.0,Petrol,Automatic
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,"₹ 830,000","1,46,000 kms • Diesel • Manual",https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida,146000.0,Diesel,Manual


In [41]:
units

{'kms'}

In [45]:
df['Fuel Type'].unique()

<StringArray>
['Petrol', 'Diesel', 'CNG', nan, 'Electric']
Length: 5, dtype: str

In [46]:
df['Type'].unique()

<StringArray>
['Automatic', 'Manual', nan]
Length: 3, dtype: str

In [54]:
df['Price'].isna().sum()

np.int64(6)

In [57]:
df =df.dropna(subset=['Price']).reset_index()

In [59]:
df.drop('index',axis = 1,inplace = True)

In [61]:
def refine_price(data):
    if type(data) == str:
        data = data.replace('₹','').replace(',','')
        return float(data)
    return data

df['Price'] = df['Price'].apply(refine_price)

In [63]:
df.drop('Information',inplace=True,axis = 1)

In [64]:
df.head()

,Title,Link,Price,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,231000.0,https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon",60000.0,Petrol,Automatic
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,575000.0,https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi",58000.0,Petrol,Manual
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,575000.0,https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi",41000.0,Petrol,Manual
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,1092000.0,https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad",53366.0,Petrol,Automatic
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,830000.0,https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida,146000.0,Diesel,Manual


In [65]:
df.to_csv('../../data/cleaned_data/cardekho_cleaned_data.csv')

## Cleaning Cars24 Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/raw_data/cars24_listings_delhi.csv')

In [4]:
df.head()

,Title,Link,Price,Del Price,Information,Image,Year,Brand,Location
0,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
1,NaN,https://www.cars24.com/buy-used-maruti-wagon-r...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
2,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
3,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
4,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN


In [8]:
df = df.dropna(subset=['Title']).reset_index()

In [10]:
df.drop(['index'],axis = 1,inplace = True)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        613 non-null    str    
 1   Link         613 non-null    str    
 2   Price        613 non-null    str    
 3   Del Price    277 non-null    str    
 4   Information  613 non-null    str    
 5   Image        613 non-null    str    
 6   Year         613 non-null    float64
 7   Brand        613 non-null    str    
 8   Location     613 non-null    str    
dtypes: float64(1), str(8)
memory usage: 43.2 KB


In [13]:
df.head()

,Title,Link,Price,Del Price,Information,Image,Year,Brand,Location
0,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 7.72 lakh,"₹ 899,000","45,498 km | Petrol | Manual | HR-98",https://media.cars24.com/hello-ar/dev/transfor...,2022.0,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
1,2019 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 5.90 lakh,"₹ 591,000","93,355 km | Petrol | Manual | DL-6C",https://media.cars24.com/hello-ar/dev/transfor...,2019.0,Maruti,Rajouri Garden
2,2024 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 8.99 lakh,"₹ 917,000","18,586 km | CNG | Manual | HR-14",https://media.cars24.com/hello-ar/dev/transfor...,2024.0,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
3,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 8.15 lakh,"₹ 931,000","33,086 km | Petrol | Auto | DL-4C",https://media.cars24.com/hello-ar/dev/transfor...,2022.0,Maruti,"Sector-18, Noida"
4,2016 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 4.50 lakh,"₹ 487,000","96,696 km | Petrol | Manual | HR-26",https://media.cars24.com/hello-ar/dev/transfor...,2016.0,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"


In [ ]:
def refine_prices(data):
    data = data.replace('₹','')
    if 'lakh' in data:
        return float(data.replace('lakh',''))*100000

df['Price'] = df['Price'].apply(refine_prices)


In [17]:
df['Year'] = df['Year'].astype(int)

In [18]:
df.head()

,Title,Link,Price,Del Price,Information,Image,Year,Brand,Location
0,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,772000.0,"₹ 899,000","45,498 km | Petrol | Manual | HR-98",https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
1,2019 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,590000.0,"₹ 591,000","93,355 km | Petrol | Manual | DL-6C",https://media.cars24.com/hello-ar/dev/transfor...,2019,Maruti,Rajouri Garden
2,2024 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,899000.0,"₹ 917,000","18,586 km | CNG | Manual | HR-14",https://media.cars24.com/hello-ar/dev/transfor...,2024,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
3,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,815000.0,"₹ 931,000","33,086 km | Petrol | Auto | DL-4C",https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"Sector-18, Noida"
4,2016 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,450000.0,"₹ 487,000","96,696 km | Petrol | Manual | HR-26",https://media.cars24.com/hello-ar/dev/transfor...,2016,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"


In [28]:
l = set()
def extract_information(rec):
    distance,fuel_type,type,region_info = rec.split('|')
    distance = int(distance.replace(',','').replace('km',''))
    if type == ' Auto ':
        type = 'Automatic'
    return distance,fuel_type.strip(),type.strip(),region_info.strip()  

df[['Kms Covered','Fuel Type','Type','Region Info']] = df['Information'].apply(extract_information).apply(pd.Series)

In [29]:
df['Fuel Type'].unique()

<StringArray>
['Petrol', 'CNG', 'Diesel', 'Hybrid']
Length: 4, dtype: str

In [30]:
df['Type'].unique()

<StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str

In [31]:
df['Region Info'].unique()

<StringArray>
['HR-98', 'DL-6C', 'HR-14', 'DL-4C', 'HR-26', 'DL-1C', 'DL-2C', 'UP-16',
 'DL-8C', 'DL-3C', 'DL-9C', 'HR-79', 'UP-80', 'HR-05', 'JH-09', 'HR-76',
 'HR-51', 'DL-12', 'HR-21', 'HR-02', 'UP-61', 'MP-07', 'HR-87', 'HR-10',
 'UP-14', 'DL-11', 'HR-19', 'DL-5C', 'HR-36', 'HR-27', 'DL-10', 'UP-15',
 'DL-7C', 'HR-06', 'HR-01', 'HR-30', 'UP-32', 'HR-50', 'BR-01', 'HR-29',
 'UK-07', 'UP-13', 'HR-33', 'HR-20', 'JK-02', 'UP-37', 'HR-89', 'UP-33',
 'DL-14', 'UP-85', 'HR-44', 'TN-10', 'RJ-32', 'JH-05', 'BR-19', '24-BH',
 'UP-63', 'CH-01', 'GJ-01', 'HR-12', 'HR-49', 'HP-68', 'WB-02', 'HR-70',
 'MH-47', 'GJ-18', 'HR-16', 'HR-85', 'DL-13', 'RJ-13', 'UP-78', 'TS-02',
 'UP-27', 'UP-21']
Length: 74, dtype: str

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        613 non-null    str    
 1   Link         613 non-null    str    
 2   Price        613 non-null    float64
 3   Del Price    277 non-null    str    
 4   Information  613 non-null    str    
 5   Image        613 non-null    str    
 6   Year         613 non-null    int64  
 7   Brand        613 non-null    str    
 8   Location     613 non-null    str    
 9   Kms Covered  613 non-null    int64  
 10  Fuel Type    613 non-null    str    
 11  Type         613 non-null    str    
 12  Region Info  613 non-null    str    
dtypes: float64(1), int64(2), str(10)
memory usage: 62.4 KB


In [33]:
df.drop(['Del Price','Information'],axis = 1,inplace = True)

In [34]:
df.head()

,Title,Link,Price,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type,Region Info
0,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,772000.0,https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"M3M Urbana, Golf Course Ext., Gurugram",45498,Petrol,Manual,HR-98
1,2019 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,590000.0,https://media.cars24.com/hello-ar/dev/transfor...,2019,Maruti,Rajouri Garden,93355,Petrol,Manual,DL-6C
2,2024 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,899000.0,https://media.cars24.com/hello-ar/dev/transfor...,2024,Maruti,"M3M Urbana, Golf Course Ext., Gurugram",18586,CNG,Manual,HR-14
3,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,815000.0,https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"Sector-18, Noida",33086,Petrol,Automatic,DL-4C
4,2016 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,450000.0,https://media.cars24.com/hello-ar/dev/transfor...,2016,Maruti,"M3M Urbana, Golf Course Ext., Gurugram",96696,Petrol,Manual,HR-26


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        613 non-null    str    
 1   Link         613 non-null    str    
 2   Price        613 non-null    float64
 3   Image        613 non-null    str    
 4   Year         613 non-null    int64  
 5   Brand        613 non-null    str    
 6   Location     613 non-null    str    
 7   Kms Covered  613 non-null    int64  
 8   Fuel Type    613 non-null    str    
 9   Type         613 non-null    str    
 10  Region Info  613 non-null    str    
dtypes: float64(1), int64(2), str(8)
memory usage: 52.8 KB


In [37]:
df.to_csv('../../data/cleaned_data/cars24_cleaned_data.csv')

## Olx Data Cleaning

In [81]:
df = pd.read_csv('../../data/raw_data/olx_luxe_listings_delhi.csv')

In [82]:
df.head()

,Title,Link,Location,Price,Information,Image,Detail,Owner
0,Porsche 911,https://www.olx.in/item/cars-c84-used-porsche-...,Rajouri Garden,"₹ 2,44,75,000",2025 - 895 km,https://apollo.olx.in:443/v1/files/xh2jctgn7oj...,"Petrol , --",1st
1,Mercedes-Benz AMG GLE Coupe,https://www.olx.in/item/cars-c84-used-mercedes...,Rajouri Garden,"₹ 1,29,00,000","2025 - 9,700 km",https://apollo.olx.in:443/v1/files/mta7txri1a2...,"Petrol , Automatic",1st
2,Land Rover Range Rover,https://www.olx.in/item/cars-c84-used-land-rov...,Vikaspuri,"₹ 1,35,75,000","2021 - 52,000 km",https://apollo.olx.in:443/v1/files/ov1ski0o9ys...,"Petrol , Automatic",1st
3,Porsche CARRERA,https://www.olx.in/item/cars-c84-used-porsche-...,Paschim Vihar,"₹ 2,29,75,000","2024 - 2,500 km",https://apollo.olx.in:443/v1/files/dbww3copx4l...,"Petrol , Automatic",1st
4,BMW X7,https://www.olx.in/item/cars-c84-used-bmw-x7-i...,Paschim Vihar,"₹ 88,00,000","2022 - 44,000 km",https://apollo.olx.in:443/v1/files/n90pxrsjnnq...,"Diesel , Automatic",1st


In [83]:
def filter_price(data):
        data = data.replace(',','').replace('₹','')
        return float(data)

df['Price'] = df['Price'].apply(filter_price)

In [84]:
df['Price']

0      24475000.0
1      12900000.0
2      13575000.0
3      22975000.0
4       8800000.0
          ...    
635     9199000.0
636    11500000.0
637     9199000.0
638     8999000.0
639    42500000.0
Name: Price, Length: 640, dtype: float64

In [85]:
def filter_information(data):
    year, distance = data.split('-')
    distance = distance.replace(',','').replace('km','')
    
    return  int(year),float(distance)

df[['Year','Kms Covered']] = df['Information'].apply(filter_information).apply(pd.Series)

In [86]:
df.head()

,Title,Link,Location,Price,Information,Image,Detail,Owner,Year,Kms Covered
0,Porsche 911,https://www.olx.in/item/cars-c84-used-porsche-...,Rajouri Garden,24475000.0,2025 - 895 km,https://apollo.olx.in:443/v1/files/xh2jctgn7oj...,"Petrol , --",1st,2025.0,895.0
1,Mercedes-Benz AMG GLE Coupe,https://www.olx.in/item/cars-c84-used-mercedes...,Rajouri Garden,12900000.0,"2025 - 9,700 km",https://apollo.olx.in:443/v1/files/mta7txri1a2...,"Petrol , Automatic",1st,2025.0,9700.0
2,Land Rover Range Rover,https://www.olx.in/item/cars-c84-used-land-rov...,Vikaspuri,13575000.0,"2021 - 52,000 km",https://apollo.olx.in:443/v1/files/ov1ski0o9ys...,"Petrol , Automatic",1st,2021.0,52000.0
3,Porsche CARRERA,https://www.olx.in/item/cars-c84-used-porsche-...,Paschim Vihar,22975000.0,"2024 - 2,500 km",https://apollo.olx.in:443/v1/files/dbww3copx4l...,"Petrol , Automatic",1st,2024.0,2500.0
4,BMW X7,https://www.olx.in/item/cars-c84-used-bmw-x7-i...,Paschim Vihar,8800000.0,"2022 - 44,000 km",https://apollo.olx.in:443/v1/files/n90pxrsjnnq...,"Diesel , Automatic",1st,2022.0,44000.0


In [87]:
df['Year'] = df['Year'].astype(int)

In [88]:
df.head()

,Title,Link,Location,Price,Information,Image,Detail,Owner,Year,Kms Covered
0,Porsche 911,https://www.olx.in/item/cars-c84-used-porsche-...,Rajouri Garden,24475000.0,2025 - 895 km,https://apollo.olx.in:443/v1/files/xh2jctgn7oj...,"Petrol , --",1st,2025,895.0
1,Mercedes-Benz AMG GLE Coupe,https://www.olx.in/item/cars-c84-used-mercedes...,Rajouri Garden,12900000.0,"2025 - 9,700 km",https://apollo.olx.in:443/v1/files/mta7txri1a2...,"Petrol , Automatic",1st,2025,9700.0
2,Land Rover Range Rover,https://www.olx.in/item/cars-c84-used-land-rov...,Vikaspuri,13575000.0,"2021 - 52,000 km",https://apollo.olx.in:443/v1/files/ov1ski0o9ys...,"Petrol , Automatic",1st,2021,52000.0
3,Porsche CARRERA,https://www.olx.in/item/cars-c84-used-porsche-...,Paschim Vihar,22975000.0,"2024 - 2,500 km",https://apollo.olx.in:443/v1/files/dbww3copx4l...,"Petrol , Automatic",1st,2024,2500.0
4,BMW X7,https://www.olx.in/item/cars-c84-used-bmw-x7-i...,Paschim Vihar,8800000.0,"2022 - 44,000 km",https://apollo.olx.in:443/v1/files/n90pxrsjnnq...,"Diesel , Automatic",1st,2022,44000.0


In [89]:
import numpy as np

In [90]:
def filter_owner(data):
    if data == '--' or data == np.nan :
        return np.nan
    elif type(data) == str:
        return data[0]
    else:
        return data
    
df['Owner'] = df['Owner'].apply(filter_owner)

In [91]:
df['Owner'].unique()

<StringArray>
['1', nan, '2', '3']
Length: 4, dtype: str

In [92]:
df["Owner"] = pd.to_numeric(df["Owner"]).astype("Int64")

In [93]:
df['Owner'].unique()

<IntegerArray>
[1, <NA>, 2, 3]
Length: 4, dtype: Int64

In [94]:
df.head()

,Title,Link,Location,Price,Information,Image,Detail,Owner,Year,Kms Covered
0,Porsche 911,https://www.olx.in/item/cars-c84-used-porsche-...,Rajouri Garden,24475000.0,2025 - 895 km,https://apollo.olx.in:443/v1/files/xh2jctgn7oj...,"Petrol , --",1,2025,895.0
1,Mercedes-Benz AMG GLE Coupe,https://www.olx.in/item/cars-c84-used-mercedes...,Rajouri Garden,12900000.0,"2025 - 9,700 km",https://apollo.olx.in:443/v1/files/mta7txri1a2...,"Petrol , Automatic",1,2025,9700.0
2,Land Rover Range Rover,https://www.olx.in/item/cars-c84-used-land-rov...,Vikaspuri,13575000.0,"2021 - 52,000 km",https://apollo.olx.in:443/v1/files/ov1ski0o9ys...,"Petrol , Automatic",1,2021,52000.0
3,Porsche CARRERA,https://www.olx.in/item/cars-c84-used-porsche-...,Paschim Vihar,22975000.0,"2024 - 2,500 km",https://apollo.olx.in:443/v1/files/dbww3copx4l...,"Petrol , Automatic",1,2024,2500.0
4,BMW X7,https://www.olx.in/item/cars-c84-used-bmw-x7-i...,Paschim Vihar,8800000.0,"2022 - 44,000 km",https://apollo.olx.in:443/v1/files/n90pxrsjnnq...,"Diesel , Automatic",1,2022,44000.0


In [99]:
df['Detail'].unique()

<StringArray>
[         'Petrol , --',   'Petrol , Automatic',   'Diesel , Automatic',
              '-- , --',       '-- , Automatic',                    nan,
      'Petrol , Manual',          'Diesel , --', 'Electric , Automatic',
      'LPG , Automatic']
Length: 10, dtype: str

In [112]:
def extract_details(data):
    
    if type(data) == str:
        fuel_type,type_ = data.split(',')
        fuel_type = fuel_type.strip()
        type_ = type_.strip()
        if fuel_type == '--':
            fuel_type = np.nan
        
        if type_ == '--':
            type_ = np.nan
        
        return fuel_type,type_
    
    return np.nan,np.nan

df[['Fuel Type','Type']] = df['Detail'].apply(extract_details).apply(pd.Series)


In [114]:
df.head()

,Title,Link,Location,Price,Information,Image,Detail,Owner,Year,Kms Covered,Fuel Type,Type
0,Porsche 911,https://www.olx.in/item/cars-c84-used-porsche-...,Rajouri Garden,24475000.0,2025 - 895 km,https://apollo.olx.in:443/v1/files/xh2jctgn7oj...,"Petrol , --",1,2025,895.0,Petrol,NaN
1,Mercedes-Benz AMG GLE Coupe,https://www.olx.in/item/cars-c84-used-mercedes...,Rajouri Garden,12900000.0,"2025 - 9,700 km",https://apollo.olx.in:443/v1/files/mta7txri1a2...,"Petrol , Automatic",1,2025,9700.0,Petrol,Automatic
2,Land Rover Range Rover,https://www.olx.in/item/cars-c84-used-land-rov...,Vikaspuri,13575000.0,"2021 - 52,000 km",https://apollo.olx.in:443/v1/files/ov1ski0o9ys...,"Petrol , Automatic",1,2021,52000.0,Petrol,Automatic
3,Porsche CARRERA,https://www.olx.in/item/cars-c84-used-porsche-...,Paschim Vihar,22975000.0,"2024 - 2,500 km",https://apollo.olx.in:443/v1/files/dbww3copx4l...,"Petrol , Automatic",1,2024,2500.0,Petrol,Automatic
4,BMW X7,https://www.olx.in/item/cars-c84-used-bmw-x7-i...,Paschim Vihar,8800000.0,"2022 - 44,000 km",https://apollo.olx.in:443/v1/files/n90pxrsjnnq...,"Diesel , Automatic",1,2022,44000.0,Diesel,Automatic


In [115]:
df.drop(['Information','Detail'],inplace=True,axis =1)

In [116]:
df.head()

,Title,Link,Location,Price,Image,Owner,Year,Kms Covered,Fuel Type,Type
0,Porsche 911,https://www.olx.in/item/cars-c84-used-porsche-...,Rajouri Garden,24475000.0,https://apollo.olx.in:443/v1/files/xh2jctgn7oj...,1,2025,895.0,Petrol,NaN
1,Mercedes-Benz AMG GLE Coupe,https://www.olx.in/item/cars-c84-used-mercedes...,Rajouri Garden,12900000.0,https://apollo.olx.in:443/v1/files/mta7txri1a2...,1,2025,9700.0,Petrol,Automatic
2,Land Rover Range Rover,https://www.olx.in/item/cars-c84-used-land-rov...,Vikaspuri,13575000.0,https://apollo.olx.in:443/v1/files/ov1ski0o9ys...,1,2021,52000.0,Petrol,Automatic
3,Porsche CARRERA,https://www.olx.in/item/cars-c84-used-porsche-...,Paschim Vihar,22975000.0,https://apollo.olx.in:443/v1/files/dbww3copx4l...,1,2024,2500.0,Petrol,Automatic
4,BMW X7,https://www.olx.in/item/cars-c84-used-bmw-x7-i...,Paschim Vihar,8800000.0,https://apollo.olx.in:443/v1/files/n90pxrsjnnq...,1,2022,44000.0,Diesel,Automatic


In [117]:
print(df['Title'])

0                       Porsche 911
1       Mercedes-Benz AMG GLE Coupe
2            Land Rover Range Rover
3                   Porsche CARRERA
4                            BMW X7
                   ...             
635               Mercedes-Benz GLS
636    Land Rover Range Rover Sport
637               Mercedes-Benz GLS
638                          BMW IX
639                     McLaren  GT
Name: Title, Length: 640, dtype: str


In [119]:
brands = ['Porsche','Mercedes-Benz','Land Rover','McLaren','BMW','Jaguar','Ferrari','Lamborgini','Bentley']


In [120]:
def get_brand(title):
    for brand in brands:
        if brand in title:
            return brand
    
    return None

df['Brand'] = df['Title'].apply(get_brand)


In [127]:
df.head()

,Title,Link,Location,Price,Image,Owner,Year,Kms Covered,Fuel Type,Type,Brand
0,Porsche 911,https://www.olx.in/item/cars-c84-used-porsche-...,Rajouri Garden,24475000.0,https://apollo.olx.in:443/v1/files/xh2jctgn7oj...,1,2025,895.0,Petrol,NaN,Porsche
1,Mercedes-Benz AMG GLE Coupe,https://www.olx.in/item/cars-c84-used-mercedes...,Rajouri Garden,12900000.0,https://apollo.olx.in:443/v1/files/mta7txri1a2...,1,2025,9700.0,Petrol,Automatic,Mercedes-Benz
2,Land Rover Range Rover,https://www.olx.in/item/cars-c84-used-land-rov...,Vikaspuri,13575000.0,https://apollo.olx.in:443/v1/files/ov1ski0o9ys...,1,2021,52000.0,Petrol,Automatic,Land Rover
3,Porsche CARRERA,https://www.olx.in/item/cars-c84-used-porsche-...,Paschim Vihar,22975000.0,https://apollo.olx.in:443/v1/files/dbww3copx4l...,1,2024,2500.0,Petrol,Automatic,Porsche
4,BMW X7,https://www.olx.in/item/cars-c84-used-bmw-x7-i...,Paschim Vihar,8800000.0,https://apollo.olx.in:443/v1/files/n90pxrsjnnq...,1,2022,44000.0,Diesel,Automatic,BMW


In [128]:
df.to_csv('../../data/cleaned_data/olx_luxe_listings.csv')

In [7]:
import pandas  as pd
import numpy as np

In [3]:
df = pd.read_csv('../../data/raw_data/olx_listings_delhi.csv')

In [5]:
df

,Title,Link,Location,Price,Information,Image,Detail,Brand
0,Mercedes-Benz S-Class,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,"₹ 1,38,00,000","2023 - 12,000 km",https://apollo.olx.in:443/v1/files/i355lkdgj62...,"Petrol , Automatic",maruti suzuki
1,Hyundai Creta,https://www.olx.in/item/cars-c84-used-hyundai-...,Pitampura,"₹ 6,40,000","2017 - 90,000 km",https://apollo.olx.in:443/v1/files/l255s5sei1e...,"Petrol , Automatic",maruti suzuki
2,Mercedes-Benz GLA,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,"₹ 42,00,000","2024 - 6,200 km",https://apollo.olx.in:443/v1/files/4dr78sfcqyj...,"Petrol , Automatic",maruti suzuki
3,Ford Endeavour,https://www.olx.in/item/cars-c84-used-ford-end...,Paschim Vihar,"₹ 18,00,000","2016 - 400,000 km",https://apollo.olx.in:443/v1/files/0k42887kzsj...,"Diesel , Automatic",maruti suzuki
4,BMW 7 Series,https://www.olx.in/item/cars-c84-used-bmw-7-se...,Rajouri Garden,"₹ 37,90,000","2016 - 55,000 km",https://apollo.olx.in:443/v1/files/nanch4yt8k8...,"Petrol , Automatic",maruti suzuki
...,...,...,...,...,...,...,...,...
11799,Kia Seltos,https://www.olx.in/item/cars-c84-used-kia-selt...,Ashok Vihar,"₹ 7,85,000","2019 - 40,000 km",https://apollo.olx.in:443/v1/files/4e45bl9zlel...,"Petrol , Manual",kia
11800,Kia Sonet,https://www.olx.in/item/cars-c84-used-kia-sone...,Kali Bari Apartment,"₹ 7,52,000","2021 - 90,210 km",https://apollo.olx.in:443/v1/files/7884zczcnuh...,"Diesel , Manual",kia
11801,Kia Seltos,https://www.olx.in/item/cars-c84-used-kia-selt...,Bakhtawarpur,"₹ 8,20,000","2021 - 23,500 km",https://apollo.olx.in:443/v1/files/n3ra6o506oo...,"Petrol , Manual",kia
11802,Kia Seltos,https://www.olx.in/item/cars-c84-used-kia-selt...,Siraspur,"₹ 7,90,000","2021 - 23,500 km",https://apollo.olx.in:443/v1/files/jq63vzkm01n...,"Petrol , Manual",kia


In [8]:
def extract_details(data):
    
    if type(data) == str:
        fuel_type,type_ = data.split(',')
        fuel_type = fuel_type.strip()
        type_ = type_.strip()
        if fuel_type == '--':
            fuel_type = np.nan
        
        if type_ == '--':
            type_ = np.nan
        
        return fuel_type,type_
    
    return np.nan,np.nan

df[['Fuel Type','Type']] = df['Detail'].apply(extract_details).apply(pd.Series)


In [9]:
df.head()

,Title,Link,Location,Price,Information,Image,Detail,Brand,Fuel Type,Type
0,Mercedes-Benz S-Class,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,"₹ 1,38,00,000","2023 - 12,000 km",https://apollo.olx.in:443/v1/files/i355lkdgj62...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic
1,Hyundai Creta,https://www.olx.in/item/cars-c84-used-hyundai-...,Pitampura,"₹ 6,40,000","2017 - 90,000 km",https://apollo.olx.in:443/v1/files/l255s5sei1e...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic
2,Mercedes-Benz GLA,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,"₹ 42,00,000","2024 - 6,200 km",https://apollo.olx.in:443/v1/files/4dr78sfcqyj...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic
3,Ford Endeavour,https://www.olx.in/item/cars-c84-used-ford-end...,Paschim Vihar,"₹ 18,00,000","2016 - 400,000 km",https://apollo.olx.in:443/v1/files/0k42887kzsj...,"Diesel , Automatic",maruti suzuki,Diesel,Automatic
4,BMW 7 Series,https://www.olx.in/item/cars-c84-used-bmw-7-se...,Rajouri Garden,"₹ 37,90,000","2016 - 55,000 km",https://apollo.olx.in:443/v1/files/nanch4yt8k8...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic


In [10]:
def filter_information(data):
    year, distance = data.split('-')
    distance = distance.replace(',','').replace('km','')
    
    return  int(year),float(distance)

df[['Year','Kms Covered']] = df['Information'].apply(filter_information).apply(pd.Series)

In [11]:
def filter_price(data):
        data = data.replace(',','').replace('₹','')
        return float(data)

df['Price'] = df['Price'].apply(filter_price)

In [15]:
df['Year'] = df['Year'].astype(int)

In [16]:
df.head()

,Title,Link,Location,Price,Information,Image,Detail,Brand,Fuel Type,Type,Year,Kms Covered
0,Mercedes-Benz S-Class,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,13800000,"2023 - 12,000 km",https://apollo.olx.in:443/v1/files/i355lkdgj62...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic,2023,12000.0
1,Hyundai Creta,https://www.olx.in/item/cars-c84-used-hyundai-...,Pitampura,640000,"2017 - 90,000 km",https://apollo.olx.in:443/v1/files/l255s5sei1e...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic,2017,90000.0
2,Mercedes-Benz GLA,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,4200000,"2024 - 6,200 km",https://apollo.olx.in:443/v1/files/4dr78sfcqyj...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic,2024,6200.0
3,Ford Endeavour,https://www.olx.in/item/cars-c84-used-ford-end...,Paschim Vihar,1800000,"2016 - 400,000 km",https://apollo.olx.in:443/v1/files/0k42887kzsj...,"Diesel , Automatic",maruti suzuki,Diesel,Automatic,2016,400000.0
4,BMW 7 Series,https://www.olx.in/item/cars-c84-used-bmw-7-se...,Rajouri Garden,3790000,"2016 - 55,000 km",https://apollo.olx.in:443/v1/files/nanch4yt8k8...,"Petrol , Automatic",maruti suzuki,Petrol,Automatic,2016,55000.0


In [17]:
df.drop(['Information','Detail','Brand'],inplace= True,axis = 1)

In [18]:
df.head()

,Title,Link,Location,Price,Image,Fuel Type,Type,Year,Kms Covered
0,Mercedes-Benz S-Class,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,13800000,https://apollo.olx.in:443/v1/files/i355lkdgj62...,Petrol,Automatic,2023,12000.0
1,Hyundai Creta,https://www.olx.in/item/cars-c84-used-hyundai-...,Pitampura,640000,https://apollo.olx.in:443/v1/files/l255s5sei1e...,Petrol,Automatic,2017,90000.0
2,Mercedes-Benz GLA,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,4200000,https://apollo.olx.in:443/v1/files/4dr78sfcqyj...,Petrol,Automatic,2024,6200.0
3,Ford Endeavour,https://www.olx.in/item/cars-c84-used-ford-end...,Paschim Vihar,1800000,https://apollo.olx.in:443/v1/files/0k42887kzsj...,Diesel,Automatic,2016,400000.0
4,BMW 7 Series,https://www.olx.in/item/cars-c84-used-bmw-7-se...,Rajouri Garden,3790000,https://apollo.olx.in:443/v1/files/nanch4yt8k8...,Petrol,Automatic,2016,55000.0


In [67]:
brands = ['Maruti Suzuki', 'Hyundai', 'Honda', 'Toyota', 'Tata', 'Mahindra', 'Mercedes-Benz','Ford','Volkswagen','Audi','Nissan' ,'BMW', 'Kia','Skoda','Renault','Land Rover','MG','Lexus','Jeep','Jaguar','Volvo','Porsche','Datsun','Fiat','Mini Cooper','BYD','Isuzu']

In [68]:
df.shape

(11804, 10)

In [69]:
def get_brand(title):
    for brand in brands:
        if brand in title:
            return brand
    
    return 'N/A'

df['Brand'] = df['Title'].apply(get_brand)

In [70]:
df['Brand'].unique()

<StringArray>
['Mercedes-Benz',       'Hyundai',          'Ford',           'BMW',
         'Skoda', 'Maruti Suzuki',        'Toyota',         'Honda',
       'Renault',      'Mahindra',          'Tata',    'Land Rover',
            'MG',    'Volkswagen',           'Kia',        'Nissan',
          'Jeep',          'Audi',        'Jaguar',         'Volvo',
          'Fiat',        'Datsun',       'Porsche',         'Lexus',
   'Mini Cooper',           'BYD',         'Isuzu']
Length: 27, dtype: str

In [72]:
df.head()

,Title,Link,Location,Price,Image,Fuel Type,Type,Year,Kms Covered,Brand
0,Mercedes-Benz S-Class,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,13800000,https://apollo.olx.in:443/v1/files/i355lkdgj62...,Petrol,Automatic,2023,12000.0,Mercedes-Benz
1,Hyundai Creta,https://www.olx.in/item/cars-c84-used-hyundai-...,Pitampura,640000,https://apollo.olx.in:443/v1/files/l255s5sei1e...,Petrol,Automatic,2017,90000.0,Hyundai
2,Mercedes-Benz GLA,https://www.olx.in/item/cars-c84-used-mercedes...,Punjabi Bagh West,4200000,https://apollo.olx.in:443/v1/files/4dr78sfcqyj...,Petrol,Automatic,2024,6200.0,Mercedes-Benz
3,Ford Endeavour,https://www.olx.in/item/cars-c84-used-ford-end...,Paschim Vihar,1800000,https://apollo.olx.in:443/v1/files/0k42887kzsj...,Diesel,Automatic,2016,400000.0,Ford
4,BMW 7 Series,https://www.olx.in/item/cars-c84-used-bmw-7-se...,Rajouri Garden,3790000,https://apollo.olx.in:443/v1/files/nanch4yt8k8...,Petrol,Automatic,2016,55000.0,BMW


In [74]:
df.columns

Index(['Title', 'Link', 'Location', 'Price', 'Image', 'Fuel Type', 'Type',
       'Year', 'Kms Covered', 'Brand'],
      dtype='str')

In [75]:
df['Fuel Type'].unique()

<StringArray>
['Petrol', 'Diesel', 'LPG', 'CNG & Hybrids', nan, 'Electric']
Length: 6, dtype: str

In [79]:
df.duplicated().sum()

np.int64(1543)

In [81]:
df = df.drop_duplicates().reset_index(drop = True)

In [82]:
df[df['Fuel Type']=='LPG']

,Title,Link,Location,Price,Image,Fuel Type,Type,Year,Kms Covered,Brand
11,Renault KWID,https://www.olx.in/item/cars-c84-used-renault-...,Paschim Vihar,465000,https://apollo.olx.in:443/v1/files/uhlbbi65wqm...,LPG,Manual,2020,99000.0,Renault
54,Renault KWID,https://www.olx.in/item/cars-c84-used-renault-...,Paschim Vihar,465000,https://apollo.olx.in:443/v1/files/uhlbbi65wqm...,LPG,Manual,2020,99000.0,Renault
7961,Nissan Sunny,https://www.olx.in/item/cars-c84-used-nissan-s...,Alwar Rural,200000,https://apollo.olx.in:443/v1/files/ammoojx42sa...,LPG,Manual,2015,105000.0,Nissan


In [76]:
df['Type'].unique()

<StringArray>
['Automatic', 'Manual', nan]
Length: 3, dtype: str

In [83]:
df.to_csv('../../data/cleaned_data/olx_listings_delhi.csv')

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/raw_data/you_drive_me_crazy_listings_delhi.csv')

In [3]:
df.head()

,Title,Link,Price,Information,Image,Model,Year,Drive,Exterior Color,Interior Color,Registration,Ownership,Fuel type
0,21 more photos,https://youdrivemecrazy.in/listings/jaguar-xj-50/,"₹3 ,500 ,000",2018 Jaguar,https://youdrivemecrazy.in/wp-content/uploads/...,Jaguar XJ 50,2018,50000 Km,Loire blue,Quilted Ivory white,HP Registered,First Owner,NaN
1,22 more photos,https://youdrivemecrazy.in/listings/porsche-ca...,"₹7 ,200 ,000",2019 Porsche,https://youdrivemecrazy.in/wp-content/uploads/...,Porsche Cayenne 3.0 V6,2019,53000 Km,Biskay blue metallic,Beige,HP Registered,First Owner,NaN
2,21 more photos,https://youdrivemecrazy.in/listings/volvo-v90-...,"₹2 ,500 ,000",2019 Volvo,https://youdrivemecrazy.in/wp-content/uploads/...,V90 Cross Country D5,2019,78000 Km,Crystal White,Amber Brown,DL Registered,First Owner,Diesel
3,22 more photos,https://youdrivemecrazy.in/listings/lexus-nx300/,"₹2 ,500 ,000",2018 Lexus,https://youdrivemecrazy.in/wp-content/uploads/...,LEXUS NX300,2018,"1,37000 Km",Sonic Quartz,NaN,HR Registered,First Owner,NaN
4,22 more photos,https://youdrivemecrazy.in/listings/ford-musta...,"₹7 ,500 ,000",2017 Ford,https://youdrivemecrazy.in/wp-content/uploads/...,FORD MUSTANG GT 5.0 V8,2017,27000 Km,Magnetic Grey Metallic,Black,HP Registered,First Owner,NaN


In [4]:
df.drop(['Title'],inplace= True,axis = 1)

In [ ]:
df['Kms Covered']

In [8]:
def filter_price(data):
        data = data.replace(',','').replace('₹','').replace(' ','')
        return float(data.strip())

df['Price'] = df['Price'].apply(filter_price)

In [18]:
def  filter_information(data):
    if type(data) == str:

        _,*brand= data.split(' ')
        
        brand =  " ".join(brand)
        if brand == 'Mini' or brand == 'cooper':
            return 'Mini Cooper'
        elif brand == 'FORD ENDEAVOUR':
            return 'Ford'
        elif brand == 'Range Rover':
            return 'Land Rover'
        return brand
    else:
        return None
    
    

df['Brand'] = df['Information'].apply(filter_information)


In [19]:
df['Brand'].unique()

<StringArray>
[     'Jaguar',     'Porsche',       'Volvo',       'Lexus',        'Ford',
        'Audi',         'BMW',  'Land Rover', 'Rolls royce',    'Mercedes',
        'Jeep', 'Mini Cooper',  'Volkswagen',           nan]
Length: 14, dtype: str

In [22]:
df = df.dropna(subset=['Brand','Ownership']).reset_index(drop = True)

In [23]:
df

,Link,Price,Information,Image,Model,Year,Drive,Exterior Color,Interior Color,Registration,Ownership,Fuel type,Brand
0,https://youdrivemecrazy.in/listings/jaguar-xj-50/,3500000.0,2018 Jaguar,https://youdrivemecrazy.in/wp-content/uploads/...,Jaguar XJ 50,2018,50000 Km,Loire blue,Quilted Ivory white,HP Registered,First Owner,NaN,Jaguar
1,https://youdrivemecrazy.in/listings/porsche-ca...,7200000.0,2019 Porsche,https://youdrivemecrazy.in/wp-content/uploads/...,Porsche Cayenne 3.0 V6,2019,53000 Km,Biskay blue metallic,Beige,HP Registered,First Owner,NaN,Porsche
2,https://youdrivemecrazy.in/listings/volvo-v90-...,2500000.0,2019 Volvo,https://youdrivemecrazy.in/wp-content/uploads/...,V90 Cross Country D5,2019,78000 Km,Crystal White,Amber Brown,DL Registered,First Owner,Diesel,Volvo
3,https://youdrivemecrazy.in/listings/lexus-nx300/,2500000.0,2018 Lexus,https://youdrivemecrazy.in/wp-content/uploads/...,LEXUS NX300,2018,"1,37000 Km",Sonic Quartz,NaN,HR Registered,First Owner,NaN,Lexus
4,https://youdrivemecrazy.in/listings/ford-musta...,7500000.0,2017 Ford,https://youdrivemecrazy.in/wp-content/uploads/...,FORD MUSTANG GT 5.0 V8,2017,27000 Km,Magnetic Grey Metallic,Black,HP Registered,First Owner,NaN,Ford
5,https://youdrivemecrazy.in/listings/porsche-pa...,9900000.0,2020 Porsche,https://youdrivemecrazy.in/wp-content/uploads/...,PORSCHE PANAMERA 4,2020,11000 km,Jet Black Metallic,NaN,HP Registered,First Owner,Petrol Car,Porsche
6,https://youdrivemecrazy.in/listings/facelift-t...,2700000.0,2021 FORD ENDEAVOUR,https://youdrivemecrazy.in/wp-content/uploads/...,FACELIFT | TITANIUM 2.0 | 4x4,2021,"1,30,000 Kms",Diamond White,Beige Leather,DL Registered,First Owner,NaN,Ford
7,https://youdrivemecrazy.in/listings/bmw-330i-g...,2550000.0,2019 BMW,https://youdrivemecrazy.in/wp-content/uploads/...,330i GT M Sport,2019,"92,000 km",Estoril Blue,Black,HR Registered,First Owner,NaN,BMW
8,https://youdrivemecrazy.in/listings/porsche-ca...,3650000.0,2013 Porsche,https://youdrivemecrazy.in/wp-content/uploads/...,Porsche Cayenne,2013,72000 Km,Moonlight Blue Metallic,Slate Grey,HR Registered,First Owner,Petrol Car,Porsche
9,https://youdrivemecrazy.in/listings/range-rove...,5800000.0,2018 Range Rover,https://youdrivemecrazy.in/wp-content/uploads/...,SPORT HSE (Facelift),2018,38000 Km,Fuji white,Beige,HR Registered,2nd Owner,NaN,Land Rover


In [35]:
import re
def filter_drive(data):
    data = re.findall(r'\d+', data)
    return int("".join(data))

df['Kms Covered'] = df['Drive'].apply(filter_drive)

In [36]:
df.head()

,Link,Price,Information,Image,Model,Year,Drive,Exterior Color,Interior Color,Registration,Ownership,Fuel type,Brand,Kms Covered
0,https://youdrivemecrazy.in/listings/jaguar-xj-50/,3500000.0,2018 Jaguar,https://youdrivemecrazy.in/wp-content/uploads/...,Jaguar XJ 50,2018,50000 Km,Loire blue,Quilted Ivory white,HP Registered,First Owner,NaN,Jaguar,50000
1,https://youdrivemecrazy.in/listings/porsche-ca...,7200000.0,2019 Porsche,https://youdrivemecrazy.in/wp-content/uploads/...,Porsche Cayenne 3.0 V6,2019,53000 Km,Biskay blue metallic,Beige,HP Registered,First Owner,NaN,Porsche,53000
2,https://youdrivemecrazy.in/listings/volvo-v90-...,2500000.0,2019 Volvo,https://youdrivemecrazy.in/wp-content/uploads/...,V90 Cross Country D5,2019,78000 Km,Crystal White,Amber Brown,DL Registered,First Owner,Diesel,Volvo,78000
3,https://youdrivemecrazy.in/listings/lexus-nx300/,2500000.0,2018 Lexus,https://youdrivemecrazy.in/wp-content/uploads/...,LEXUS NX300,2018,"1,37000 Km",Sonic Quartz,NaN,HR Registered,First Owner,NaN,Lexus,137000
4,https://youdrivemecrazy.in/listings/ford-musta...,7500000.0,2017 Ford,https://youdrivemecrazy.in/wp-content/uploads/...,FORD MUSTANG GT 5.0 V8,2017,27000 Km,Magnetic Grey Metallic,Black,HP Registered,First Owner,NaN,Ford,27000


In [37]:
def filter_owner(data):
    if data == 'First Owner':
        return 1
    elif data == '2nd Owner':
        return 2
    elif data == '3rd Owner':
        return 3
    else:
        return np.nan
df['Ownership'] = df['Ownership'].apply(filter_owner)


In [38]:
df.head()

,Link,Price,Information,Image,Model,Year,Drive,Exterior Color,Interior Color,Registration,Ownership,Fuel type,Brand,Kms Covered
0,https://youdrivemecrazy.in/listings/jaguar-xj-50/,3500000.0,2018 Jaguar,https://youdrivemecrazy.in/wp-content/uploads/...,Jaguar XJ 50,2018,50000 Km,Loire blue,Quilted Ivory white,HP Registered,1,NaN,Jaguar,50000
1,https://youdrivemecrazy.in/listings/porsche-ca...,7200000.0,2019 Porsche,https://youdrivemecrazy.in/wp-content/uploads/...,Porsche Cayenne 3.0 V6,2019,53000 Km,Biskay blue metallic,Beige,HP Registered,1,NaN,Porsche,53000
2,https://youdrivemecrazy.in/listings/volvo-v90-...,2500000.0,2019 Volvo,https://youdrivemecrazy.in/wp-content/uploads/...,V90 Cross Country D5,2019,78000 Km,Crystal White,Amber Brown,DL Registered,1,Diesel,Volvo,78000
3,https://youdrivemecrazy.in/listings/lexus-nx300/,2500000.0,2018 Lexus,https://youdrivemecrazy.in/wp-content/uploads/...,LEXUS NX300,2018,"1,37000 Km",Sonic Quartz,NaN,HR Registered,1,NaN,Lexus,137000
4,https://youdrivemecrazy.in/listings/ford-musta...,7500000.0,2017 Ford,https://youdrivemecrazy.in/wp-content/uploads/...,FORD MUSTANG GT 5.0 V8,2017,27000 Km,Magnetic Grey Metallic,Black,HP Registered,1,NaN,Ford,27000


In [39]:
df.drop(['Information','Drive'],inplace= True,axis = 1)

In [40]:
df.head()

,Link,Price,Image,Model,Year,Exterior Color,Interior Color,Registration,Ownership,Fuel type,Brand,Kms Covered
0,https://youdrivemecrazy.in/listings/jaguar-xj-50/,3500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,Jaguar XJ 50,2018,Loire blue,Quilted Ivory white,HP Registered,1,NaN,Jaguar,50000
1,https://youdrivemecrazy.in/listings/porsche-ca...,7200000.0,https://youdrivemecrazy.in/wp-content/uploads/...,Porsche Cayenne 3.0 V6,2019,Biskay blue metallic,Beige,HP Registered,1,NaN,Porsche,53000
2,https://youdrivemecrazy.in/listings/volvo-v90-...,2500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,V90 Cross Country D5,2019,Crystal White,Amber Brown,DL Registered,1,Diesel,Volvo,78000
3,https://youdrivemecrazy.in/listings/lexus-nx300/,2500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,LEXUS NX300,2018,Sonic Quartz,NaN,HR Registered,1,NaN,Lexus,137000
4,https://youdrivemecrazy.in/listings/ford-musta...,7500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,FORD MUSTANG GT 5.0 V8,2017,Magnetic Grey Metallic,Black,HP Registered,1,NaN,Ford,27000


In [41]:
df['Fuel type'].unique()

<StringArray>
[nan, 'Diesel', 'Petrol Car']
Length: 3, dtype: str

In [42]:
def filter_fuelType(data):
    if data == 'Petrol Car':
        return 'Petrol'
    return data

df['Fuel Type'] = df['Fuel type'].apply(filter_fuelType)

In [43]:
df.drop('Fuel type',axis = 1,inplace = True)

In [44]:
df.head()

,Link,Price,Image,Model,Year,Exterior Color,Interior Color,Registration,Ownership,Brand,Kms Covered,Fuel Type
0,https://youdrivemecrazy.in/listings/jaguar-xj-50/,3500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,Jaguar XJ 50,2018,Loire blue,Quilted Ivory white,HP Registered,1,Jaguar,50000,NaN
1,https://youdrivemecrazy.in/listings/porsche-ca...,7200000.0,https://youdrivemecrazy.in/wp-content/uploads/...,Porsche Cayenne 3.0 V6,2019,Biskay blue metallic,Beige,HP Registered,1,Porsche,53000,NaN
2,https://youdrivemecrazy.in/listings/volvo-v90-...,2500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,V90 Cross Country D5,2019,Crystal White,Amber Brown,DL Registered,1,Volvo,78000,Diesel
3,https://youdrivemecrazy.in/listings/lexus-nx300/,2500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,LEXUS NX300,2018,Sonic Quartz,NaN,HR Registered,1,Lexus,137000,NaN
4,https://youdrivemecrazy.in/listings/ford-musta...,7500000.0,https://youdrivemecrazy.in/wp-content/uploads/...,FORD MUSTANG GT 5.0 V8,2017,Magnetic Grey Metallic,Black,HP Registered,1,Ford,27000,NaN


In [46]:
df.to_csv('../../data/cleaned_data/you_drive_me_crazy_listings.csv')

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/raw_data/fusioncars_data.csv')

In [3]:
df.head()

,Title,Registered,Fuel,Kms,Price,Image_URL,Sold_Status,URL
0,2022/23 LAND ROVER DEFENDER 2.0 HSE 110,2023.0,Petrol,34000.0,"₹ 98,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2025-range-rover-ve...
1,2015/16 LAMBORGHINI HURACAN COUPE LP 610-4,2016.0,Petrol,18000.0,"₹ 2,38,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2015-16-lamborghini...
2,2022/23 MINI COOPER SE 3 DOOR,2023.0,Electric,5000.0,"₹ 34,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2017-18-mini-cooper...
3,2024/25 TOYOTA LAND CRUISER 300,2025.0,Petrol,7000.0,"₹ 2,35,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2023-toyota-fortune...
4,2022 MERCEDES BENZ EQS 53 AMG 4MATIC+,2022.0,Electric,9000.0,"₹ 1,25,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2024-26-mercedes-be...


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        477 non-null    str    
 1   Registered   465 non-null    float64
 2   Fuel         465 non-null    str    
 3   Kms          465 non-null    float64
 4   Price        465 non-null    str    
 5   Image_URL    477 non-null    str    
 6   Sold_Status  477 non-null    str    
 7   URL          477 non-null    str    
dtypes: float64(2), str(6)
memory usage: 29.9 KB


In [12]:
df['Registered'].unique()

array([2023., 2016., 2025., 2022., 2026., 2018., 2019., 2021., 2015.,
       2020., 2024., 2017.,   nan])

In [15]:
df['Year'] = df['Registered'].astype('Int64',errors='ignore')
df['Kms Covered'] = df['Kms']

In [16]:
df.head()

,Title,Registered,Fuel,Kms,Price,Image_URL,Sold_Status,URL,Year,Kms Covered
0,2022/23 LAND ROVER DEFENDER 2.0 HSE 110,2023.0,Petrol,34000.0,"₹ 98,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2025-range-rover-ve...,2023,34000.0
1,2015/16 LAMBORGHINI HURACAN COUPE LP 610-4,2016.0,Petrol,18000.0,"₹ 2,38,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2015-16-lamborghini...,2016,18000.0
2,2022/23 MINI COOPER SE 3 DOOR,2023.0,Electric,5000.0,"₹ 34,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2017-18-mini-cooper...,2023,5000.0
3,2024/25 TOYOTA LAND CRUISER 300,2025.0,Petrol,7000.0,"₹ 2,35,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2023-toyota-fortune...,2025,7000.0
4,2022 MERCEDES BENZ EQS 53 AMG 4MATIC+,2022.0,Electric,9000.0,"₹ 1,25,00,000","data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2024-26-mercedes-be...,2022,9000.0


In [17]:
df['Fuel'].unique()

<StringArray>
['Petrol', 'Electric', 'Diesel', 'Hybrid', nan]
Length: 5, dtype: str

In [20]:
def filter_price(data):
    if type(data) == str:
        data = data.replace(',','').replace('₹','')
        return float(data)
    return data


In [22]:
df['Price'] = df['Price'].apply(filter_price)

In [23]:
df.head()

,Title,Registered,Fuel,Kms,Price,Image_URL,Sold_Status,URL,Year,Kms Covered
0,2022/23 LAND ROVER DEFENDER 2.0 HSE 110,2023.0,Petrol,34000.0,9800000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2025-range-rover-ve...,2023,34000.0
1,2015/16 LAMBORGHINI HURACAN COUPE LP 610-4,2016.0,Petrol,18000.0,23800000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2015-16-lamborghini...,2016,18000.0
2,2022/23 MINI COOPER SE 3 DOOR,2023.0,Electric,5000.0,3400000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2017-18-mini-cooper...,2023,5000.0
3,2024/25 TOYOTA LAND CRUISER 300,2025.0,Petrol,7000.0,23500000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2023-toyota-fortune...,2025,7000.0
4,2022 MERCEDES BENZ EQS 53 AMG 4MATIC+,2022.0,Electric,9000.0,12500000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",Sold,https://fusioncars.in/cars/2024-26-mercedes-be...,2022,9000.0


In [25]:
df.drop(['Registered','Kms','Sold_Status'],inplace= True,axis = 1)

In [26]:
df.head()

,Title,Fuel,Price,Image_URL,URL,Year,Kms Covered
0,2022/23 LAND ROVER DEFENDER 2.0 HSE 110,Petrol,9800000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",https://fusioncars.in/cars/2025-range-rover-ve...,2023,34000.0
1,2015/16 LAMBORGHINI HURACAN COUPE LP 610-4,Petrol,23800000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",https://fusioncars.in/cars/2015-16-lamborghini...,2016,18000.0
2,2022/23 MINI COOPER SE 3 DOOR,Electric,3400000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",https://fusioncars.in/cars/2017-18-mini-cooper...,2023,5000.0
3,2024/25 TOYOTA LAND CRUISER 300,Petrol,23500000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",https://fusioncars.in/cars/2023-toyota-fortune...,2025,7000.0
4,2022 MERCEDES BENZ EQS 53 AMG 4MATIC+,Electric,12500000.0,"data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0...",https://fusioncars.in/cars/2024-26-mercedes-be...,2022,9000.0


In [27]:
df.to_csv('../../data/cleaned_data/fusion_cars_listings.csv')